# **Pibic-Pibiti-Huac**: Arquitetura geral para _fine-tuning_ supervisionado de _LLMs_ de código aberto (_open-source_)

Este notebook faz parte de um projeto de pesquisa PIBIC/PIBITI, desenvolvido na Universidade Federal de Campina Grande (UFCG), com foco em conceber e avaliar uma ferramenta, baseda em Inteligência Artificial, para classificação e geração de laudos Radiografia de Tórax produzidos pelo setor de Diagnóstico por Imagem do Hospital Universitário Alcides Carneiro (HUAC).

## Propósito deste _notebook_

O foco deste notebook é construir uma arquitetura geral de _Fine-Tuning_ supervisionado para modelos de linguagem de grande escala (_LLMs_) de código aberto, cuja seleção se deu em uma etapa anterior desta mesma pesquisa.

## Metodologia

Pretende-se utilizar, aqui, técnicas de _Fine-Tuning_ eficiente, como _Low-Rank Adaptation_ (LoRa) e _quantização_.

## Dataset utilizado

Em etapas anteriores desta pesquisa, desenvolvemos _datasets_ que abarcam laudos em imagens advindas de exames de Radiografia de Tórax, devidamente **tratados** e **anonimizados**.

Nesse sentido, em se tratanto de **aprendizagem supervisionada**, utilizaremos um _dataset_ de laudos rotulados segundo três categorias:

 - **OURO**.
 - **PRATA**.
 - **BRONZE**.

Cada categoria reflete, em níveis distintos, o grau de refinamento estrutual e técnico de laudos de Radiografia de Tórax.

O dataset a ser utilizado neste notebook pode ser visualizado [aqui](https://huggingface.co/datasets/guilhermenf/huac_labeled_chest_xray_reports).

## Outros _datasets_

 - [_Dataset_ apenas com laudos de Radiografia de Tórax.](https://huggingface.co/datasets/guilhermenf/huac_chest_xray_reports)
 - [_Dataset_ com laudos e imagens (PA e PERFIL) de Radiografa de Tórax.](https://huggingface.co/datasets/guilhermenf/huac_chest_xray_reports_images)

## Autores

**Guilherme Noronha**, graduando em Ciência da Computação pela Universidade Federal de Campina Grande (UFCG).

 - **Email**: guilherme.noronha.fragoso@ccc.ufcg.edu.br
 - **Github**: [guinoronhaf](https://github.com/guinoronhaf)
 - **HuggingFace**: [guilhermenf](https://huggingface.co/guilhermenf)
 - **Linkedin**: [Guilherme Fragoso](https://www.linkedin.com/in/guilherme-noronha-fragoso/)

**João Ventura**, graduando em Ciência da Computação pela Universidade Federal de Campina Grande (UFCG).

 - **Email**: joao.ventura.crispim.neto@ccc.ufcg.edu.br
 - **Github**: [joaoneto9](https://github.com/joaoneto9)
 - **HuggingFace**: [joaneto9](https://huggingface.co/joaoneto9)
 - **Linkedin**: [João Neto](https://www.linkedin.com/in/jo%C3%A3oneto09/)

## Instalando dependências

In [ ]:
# Instala ferramentas necessárias para realização de fine-tuning eficiente supervisionado
# google-cloud-bigquery e sacremoses são necessárias para carregamento de alguns dos modelos selecionados
!pip install -q -U transformers peft accelerate bitsandbytes trl google-cloud-bigquery sacremoses

print("Dependências instaladas!")

## Login no _HuggingFace_

Antes de tudo, é preciso fazer login, utilizando token de acesso, na plataforma _HuggingFace_, tanto para baixar os modelos quanto para fazer download do dataset de treino.

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN_GUI") # Obtem HF token do sistema de Secrets do Kaggle

try:
    login(hf_token) # Tenta realizar login no HuggingFace
    print("Login realizado!")
except Exception as exc: # Exceçao em caso de insucesso
    print(f"Algo deu errado: {exc}")

## Definindo **catálogo de modelos**

Nessa etapa, definiremos, utilizando um dicionário Python, quais serão os modelos efetivamente treinados.

In [ ]:
models_dict = {
    "mistral-7b": { # id do modelo
        "path": "mistralai/Mistral-7B-Instruct-v0.3", # url do modelo
        "remote_code": True # acesso a remote_code durante carregamento
    },
    "llama-1b": {
        "path": "meta-llama/Llama-3.2-1B-Instruct",
        "remote_code": True
    },
    "llama-8b": {
        "path": "meta-llama/Llama-3.1-8B-Instruct",
        "remote_code": True
    },
    "llama-3b": {
        "path": "meta-llama/Llama-3.2-3B-Instruct",
        "remote_code": True
    },
    "phi-3-mini-4b": {
        "path": "microsoft/Phi-3-mini-4k-instruct",
        "remote_code": False
    },
    "qwen2.5-3b": {
        "path": "Qwen/Qwen2.5-3B-Instruct",
        "remote_code": True
    },
    "biogpt-large": {
        "path": "microsoft/BioGPT-Large",
        "remote_code": True
    },
    "biomedical-llama-1b": {
        "path": "ContactDoctor/Bio-Medical-Llama-3-2-1B-CoT-012025",
        "remote_code": True
    }
}

print("Catálogo de modelos definido!")

In [ ]:
# Define a lista de modelos possíveis
models_to_chose = list(models_dict.keys())

print(models_to_chose)

In [ ]:
# Define um modelo para ser alvo do treinamento
model_id = models_to_chose[6]

print(f"Modelo escolhido: {model_id}")

In [ ]:
model_path = models_dict[model_id]["path"] # Extrai a url do modelo no HuggingFace
model_remote_code = models_dict[model_id]["remote_code"] # Determina se remote_code será ou não utilizado

## Configurações de treinamento

Agora, com o catálogo de modelos definido, determinaremos as configurações do treinamento visando eficiência: **LoRa** e **quantização**.

### Quantização

Para determinar as configurações de **quantização**, utilizaremos a classe `BitsAndBytesConfig`, disponível via bibliotecas _transformers_ e _bitsandbytes_.

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Configurações de quantização finalizadas!")

## Carregando modelo e _tokenizer_

Nessa etapa, baixaremos arquivos do repositório do modelo no _HuggingFace_.

### Modelo

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=model_remote_code
)

print("Modelo baixado!")

### _Tokenizer_

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
DEFAULT_CHAT_TEMPLATE = """
{% for message in messages %}
{% if message['role'] == 'system' %}
<System>
{{ message['content'] }}

{% elif message['role'] == 'user' %}
<User>
{{ message['content'] }}

{% elif message['role'] == 'assistant' %}
<Assistant>
{{ message['content'] }}{{ eos_token }}

{% endif %}
{% endfor %}

{% if add_generation_prompt %}
<Assistant>
{% endif %}
"""

print("Chat-template padrão definido!")

In [ ]:
if tokenizer.chat_template is None:
    tokenizer.chat_template = DEFAULT_CHAT_TEMPLATE
    print("chat_template default setado!")
else:
    print("Modelo já possui chat_template!")

## Aplicando configurações de _LoRa_

Agora definiremos as configurações de LoRa. Para isso, será preciso definir, para o modelo em questão, quais são os módulos-alvo de treinamento.

### Obtendo **target modules**

In [ ]:
import bitsandbytes as bnb
import torch

"""
Obtém os target modules do fine tuning um modelo específico, no formato de lista.

params:
    model: modelo a ter os target modules extraídos
returns:
    lista com os target modules.
"""
def find_all_linear_names(model) -> list():
    cls = bnb.nn.Linear4bit

    lora_module_names = set()

    for name, module in model.named_modules():
        if isinstance(module, cls):
            name_sp = name.split('.')
            lora_module_names.add(name_sp[-1] if len(name_sp) > 1 else name)

    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')

    return list(lora_module_names)


print("Função para definição dos target modules definida!")

In [ ]:
model_target_modules = find_all_linear_names(model) # target modules obtidos

In [ ]:
print(model_target_modules)

### Configurando _LoRa_

In [ ]:
import gc
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ----- limpando cache das GPUs -----
gc.collect()
torch.cuda.empty_cache()
# -------------

model = prepare_model_for_kbit_training(model) # Prepara modelo para que o fine tuning quantizado funcione corretamente

# ----- Definição das configurações de LoRa -----
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=model_target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" # tarefa a ser desempenhada
)

In [ ]:
model = get_peft_model(model, peft_config) # modelo preparado para fine tuning eficiente

In [ ]:
print("Parâmetros treináveis:")
model.print_trainable_parameters()

## Preparando _dataset_ de treinamento

Como exposto anteriormente, utilizaremos o _dataset_ de **laudos categorizados** disponível neste [link.](https://huggingface.co/datasets/guilhermenf/huac_labeled_chest_xray_reports)

### Carregando _dataset_

In [ ]:
from datasets import load_dataset, Dataset

# Path do dataset
dataset_path = "guilhermenf/huac_labeled_chest_xray_reports"

# Carrega o dataset completo
ds = load_dataset(dataset_path)

ds_train = ds["train"] # Dataset para treino
ds_test = ds["test"] # Dataset para teste

### Criando amostras de dados a serem injetadas no treinamento

Nessa etapa, criaremos uma função, chamada `crate_sample()`, que será responsável por receber um conjunto de elementos do dataset e convertê-lo em uma estrutura adequada para injeção no modelo durante o treinamento.

In [ ]:
system_prompt = """
Você é um Radiologista Analista Sênior especializado em auditoria estrutural de laudos radiológicos.

Você deve avaliar a QUALIDADE ESTRUTURAL de um laudo de Radiografia de Tórax.

Você deve retornar um JSON como resposta.

Avalie:
- cobertura estrutural do laudo;
- formalismo técnico;
- especificidade anatômica;
- completude de descrição.

Estrura da avaliação:
- Você deve atribuir uma classificação (label) ao laudo -> OURO | PRATA | BRONZE.
- Você deve justificar sua classificação a partir de elementos do laudo -> feedback.

Estrutura da resposta:
{{
    "label": OURO | PRATA | BRONZE,
    "feedback": justificativa para a classificação (label) 
}}

Não avalie diagnóstico médico.
"""

In [ ]:
import json

"""
Recebe um conjunto de dados do dataset, e tokeniza os dados de cada um dos elementos utilizando o tokenizer do modelo em questão.

params:
    sample: conjunto de dados.
returns:
    dicionáiro contendo dados devidamente tokenizados.
"""
def create_sample(sample) -> dict():
    global tokenizer
    # ---- Obtém colunas do dataset para treinamento supervisionado ----
    report_content = sample["report"]
    report_label = sample["label"]
    report_feedback = sample["feedback"]
    # ----
    texts = []

    for report, label, feedback in zip(report_content, report_label, report_feedback): # Associa cada informação a um laudo
        assistant_content = json.dumps({ # Gera uma string, representando um JSON, que contém as informações de classificação (label) e feedback do laudo.
            "label": label,
            "feedback": feedback
        }, ensure_ascii=False)
        
        chat = [
            {"role": "system", "content": system_prompt}, # system_prompt (Instruções)
            {"role": "user", "content": report}, # user_prompt (Laudo)
            {"role": "assistant", "content": assistant_content} # assistant (Label e feedback)
        ]

        tokenized_data = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False) + tokenizer.eos_token # Aplica chat_template ao chat
        texts.append(tokenized_data) # Adiciona chat tokenizado à lista texts

    return { "text": texts }


print("Função para tokenização do dataset definida!")

In [ ]:
dataset_train = ds_train.map(create_sample, batched=True, remove_columns=ds_train.column_names) # Mapeia o dataset, aplicando a função definida anteriormente

In [ ]:
dataset_train = dataset_train.select_columns(["text"]) # Filtra apenas a coluna de interesse ("text")

In [ ]:
print(dataset_train[0]) # Exemplo do primeiro elemento do dataset tokenizado segundo o chat_template

## Treinamento do modelo

Agora, iremos de fato realizar o treinamento do modelo em questão utilizando o _dataset_ de treino recém-gerado. Nesta etapa, utilizaremos a técnica de **Fine-Tuning Supervisionado** (_Supervisioned Fine-Tuning_), em que injetamos no modelo os dados **rotulados**. Nesse caso, a rotulação se dá a partir da **Label**, isto é, a classificação estrutural de cada laudo em **OURO**, **PRATA** ou **BRONZE**.

### Definição dos parâmetros de treinamento

In [ ]:
output_dir = f"/kaggle/working/{model_id}_fine_tuning/"

In [ ]:
repo_id = f"guilhermenf/chest_xray_reports_labeling_finetuning_{model_id}"

In [ ]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    num_train_epochs=3.0,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    gradient_accumulation_steps=16,
    logging_strategy="steps",
    logging_steps=1,
    gradient_checkpointing=True,
    push_to_hub=True,
    hub_model_id=repo_id,
    hub_token=hf_token,
    hub_private_repo=False
)

print("Configurações de treinamento definidas!")

### Realizando treinamento

In [ ]:
import torch
from trl import SFTTrainer

torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train
)

print("Configurações finais de treinamento feitas!")

trainer.train()

In [ ]:
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

## Recarregando modelo e injetando adaptadores _LoRa_

Apesar de o modelo estar carregado em memória, através da variável `model`, utilizaremos uma abordagem mais técnica do ponto de vista de _LoRa_.

Carregaremos o modelo base do _HuggingFacep_, novamente. Em seguinda, injetaremos os adaptadores _LoRa_ obtivos com a realização do **Fine-Tuning supervisionado**. Esse processo reflete justamente o _pipeline_ adequado para utilização de técnicas de **Fine-Tuning eficiente** como _LoRa_.

In [ ]:
# Deleta objetos antes do carregamento
del model
del tokenizer
del trainer

print("Objetos deletados!")

In [ ]:
import gc
import torch

# Limpa cache das GPUs
gc.collect()
torch.cuda.empty_cache()

print("Garbage Collector invocado e cache limpo!")

In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Configurações de quantização finalizadas!")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(repo_id)

print("Tokenizer recarregado!")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=model_remote_code
)

print("Modelo baixado!")

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    repo_id
)

print("Adaptadores LoRa acoplados!")

In [ ]:
model.eval()

## Realizando inferências

Aqui, realizaremos algumas inferências pontuais para testar o desempenho do modelo, em caráter preliminar.

In [ ]:
# Obtém elemento inicial do dataset de treino
test_ds_element = ds_test[2]

print(test_ds_element)

In [ ]:
test_chat = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_ds_element["report"]}
]

print("Chat de teste definido!")

In [ ]:
text = tokenizer.apply_chat_template(
    test_chat,
    tokenize=False,
    add_generation_prompt=True
)

print("Aplicação do chat_template concluída!")

In [ ]:
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

print("Tokenização aplicada!")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)

print("Geração da resposta concluída!")

In [ ]:
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(response)